In [1]:
# Cell 1 — imports and paths
import json
import numpy as np
import torch
from pathlib import Path
from scipy.stats import norm
from sklearn.decomposition import PCA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

PROJECT_ROOT = Path(r"C:\Users\shlok\projects\ddp-llm")
SCENERY = PROJECT_ROOT / "scenery-search"
DATA = SCENERY / "data"

PARSER_BASE = "Qwen/Qwen2.5-3B-Instruct"
PARSER_ADAPTER = PROJECT_ROOT / "checkpoints" / "qwen3b-qlora-v1" / "checkpoint-240"
VERBALIZER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

SIGMA_EPS = 0.05

In [2]:
# Cell 2 — load scenery data (same as viz notebook)
data = np.load(DATA / "scenery_embedding.npz", allow_pickle=True)
X = data["E_work"]
X_scale = X.std(axis=0).mean()
X = X / X_scale
paths = data["paths"]
labels = data["labels"]
n, d = X.shape
print(f"loaded {n} items, working dim = {d}, X_scale = {X_scale:.4f}")

loaded 600 items, working dim = 10, X_scale = 0.1250


In [3]:
# Cell 3 — GAUSSSEARCH primitives (copy from your existing notebook)
def symmetrize(A):
    return 0.5 * (A + A.T)

def bisecting_hyperplane(x_i, x_j):
    diff = x_i - x_j
    nrm = np.linalg.norm(diff)
    if nrm < 1e-12:
        w = np.zeros_like(diff); w[0] = 1.0
        return w, 0.0
    w = diff / nrm
    b = (np.dot(x_j, x_j) - np.dot(x_i, x_i)) / (2 * nrm)
    return w, b

def probit_prob(x_i, x_j, x_t, sigma_eps):
    w, b = bisecting_hyperplane(x_i, x_j)
    return norm.cdf((np.dot(x_t, w) + b) / sigma_eps)

def query_oracle(x_i, x_j, x_t, sigma_eps, rng):
    p = probit_prob(x_i, x_j, x_t, sigma_eps)
    return 0 if rng.random() < p else 1

def sample_mirror(mu, Sigma, X, used, rng):
    Sigma = symmetrize(Sigma)
    eigvals, eigvecs = np.linalg.eigh(Sigma)
    w_star = eigvecs[:, -1]
    b_star = -np.dot(w_star, mu)
    z1 = rng.multivariate_normal(mu, Sigma)
    z2 = z1 - 2 * (np.dot(w_star, z1) + b_star) * w_star
    def nearest(z, exclude):
        diffs = X - z
        d2 = np.einsum('nd,nd->n', diffs, diffs)
        for k in exclude:
            d2[k] = np.inf
        return int(np.argmin(d2))
    i = nearest(z1, used)
    j = nearest(z2, used | {i})
    return i, j

def adf_update(mu, Sigma, x_i, x_j, y, sigma_eps):
    w, b = bisecting_hyperplane(x_i, x_j)
    if y == 1:
        w = -w; b = -b
    Sw = Sigma @ w
    wSw = float(w @ Sw)
    denom = np.sqrt(wSw + sigma_eps**2)
    mu_w = float(mu @ w)
    g = (mu_w + b) / denom
    phi_g = norm.pdf(g)
    Phi_g = max(norm.cdf(g), 1e-12)
    alpha = phi_g / (Phi_g * denom)
    beta = -alpha * (g / denom + alpha)
    tau = -beta / (1 + beta * wSw)
    nu = (alpha - beta * (mu_w + b)) / (1 + beta * wSw)
    Sigma_new = Sigma - (tau / (1 + tau * wSw)) * np.outer(Sw, Sw)
    Sigma_new = symmetrize(Sigma_new)
    mu_new = mu + (nu - b * tau - tau * mu_w) * (Sigma_new @ w)
    return mu_new, Sigma_new

In [4]:
# Cell 4 — load parser (Qwen 3B + LoRA adapter), 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

parser_tok = AutoTokenizer.from_pretrained(PARSER_BASE)
if parser_tok.pad_token is None:
    parser_tok.pad_token = parser_tok.eos_token

parser_base = AutoModelForCausalLM.from_pretrained(
    PARSER_BASE, quantization_config=bnb_config, device_map="cuda",
)
parser = PeftModel.from_pretrained(parser_base, str(PARSER_ADAPTER))
parser.eval()
print(f"parser loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

W0806 23:03:50.848000 7104 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


parser loaded. VRAM: 2.18 GB


In [5]:
# Cell 5 — load verbalizer (Qwen 1.5B), fp16 for speed
verb_tok = AutoTokenizer.from_pretrained(VERBALIZER_MODEL)
if verb_tok.pad_token is None:
    verb_tok.pad_token = verb_tok.eos_token

verbalizer = AutoModelForCausalLM.from_pretrained(
    VERBALIZER_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda",
)
verbalizer.eval()
print(f"verbalizer loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"verbalizer dtype: {verbalizer.dtype}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

verbalizer loaded. VRAM: 5.27 GB
verbalizer dtype: torch.float16


In [19]:
# Cell 6 — prompts + generation helpers (no rep penalty, cleaner prompt)
PARSER_SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""


STYLES = [
    ("letter",     "Just say the letter: 'A' or 'B'. Nothing else."),
    ("ordinal",    "Say 'the first one' if they picked A, or 'the second one' if they picked B. Nothing else."),
    ("positional", "Say 'the left one' if they picked A, or 'the right one' if they picked B. Nothing else."),
    ("direct",     "Commit clearly to their choice in one short sentence. Examples: 'Option A.', 'I'll go with B.', 'A for sure.'"),
    ("casual",     "Casual and short, 3-6 words, but commit clearly. Examples: 'A works', 'B for me', 'gimme A'."),
]


def verbalize(picked_letter, rng):
    style_name, style_instruction = STYLES[rng.integers(0, len(STYLES))]
    other = "B" if picked_letter == "A" else "A"
    messages = [
        {"role": "system", "content":
            f"You are simulating a user who has just picked option {picked_letter} out of two options A and B. "
            f"They did NOT pick {other}. "
            f"Reminder: A is the first / left option. B is the second / right option. "
            f"Style: {style_instruction} "
            f"Output only the user's response, nothing else."},
        {"role": "user", "content": "Response:"},
    ]
    inputs = verb_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    torch.manual_seed(int(rng.integers(0, 1_000_000)))
    with torch.no_grad():
        out = verbalizer.generate(
            **inputs, max_new_tokens=15,
            do_sample=True, temperature=0.8, top_p=0.9,
            pad_token_id=verb_tok.eos_token_id,
        )
    text = verb_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip().strip('"').strip("'"), style_name


def parse(utterance, options=("A", "B")):
    options_str = "[" + ", ".join(f'"{o}"' for o in options) + "]"
    user_msg = f"Options: {options_str}\nUser: {utterance}"
    messages = [
        {"role": "system", "content": PARSER_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    inputs = parser_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = parser.generate(
            **inputs, max_new_tokens=10, do_sample=False,
            pad_token_id=parser_tok.eos_token_id,
        )
    text = parser_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


# device + dtype check
print(f"parser device:     {next(parser.parameters()).device}, dtype: {parser.dtype}")
print(f"verbalizer device: {next(verbalizer.parameters()).device}, dtype: {verbalizer.dtype}")
print(f"VRAM used:         {torch.cuda.memory_allocated()/1e9:.2f} GB")

# smoke test
smoke_rng = np.random.default_rng(seed=7)
for _ in range(7):
    for y in [0, 1]:
        picked = "A" if y == 0 else "B"
        utt, actual_style = verbalize(picked, smoke_rng)
        parsed = parse(utt)
        print(f"style={actual_style:11s} y={y} picked={picked}  utt={utt!r:50s}  parsed={parsed!r}")

parser device:     cuda:0, dtype: torch.bfloat16
verbalizer device: cuda:0, dtype: torch.float16
VRAM used:         5.29 GB
style=casual      y=0 picked=A  utt='Works great!'                                      parsed=['A']
style=direct      y=1 picked=B  utt="I'll go with Option B."                            parsed=['B']
style=positional  y=0 picked=A  utt='the left one'                                      parsed=['A']
style=casual      y=1 picked=B  utt='B for sure!'                                       parsed=['B']
style=letter      y=0 picked=A  utt='A'                                                 parsed=['A']
style=ordinal     y=1 picked=B  utt='the second one'                                    parsed=['B']
style=casual      y=0 picked=A  utt='Works fine.'                                       parsed=['A']
style=positional  y=1 picked=B  utt='The right one.'                                    parsed=['A']
style=letter      y=0 picked=A  utt='A'                             

In [21]:
# Cell 7 — closed-loop search
def subset_to_y(parsed, options=("A", "B")):
    """Convert parser output to a y value.
    Returns (y, status) where status is one of: 'clean', 'reject_all', 'uncertain', 'both', 'malformed'.
    y is None when the step should be skipped.
    """
    if parsed is None:
        return None, "malformed"
    if parsed == "*":
        return None, "uncertain"
    if parsed == []:
        return None, "reject_all"
    if not isinstance(parsed, list):
        return None, "malformed"
    if len(parsed) == 2:
        return None, "both"
    if len(parsed) == 1 and parsed[0] in options:
        return options.index(parsed[0]), "clean"
    return None, "malformed"


def closed_loop_search(X, target_idx, sigma_eps, max_queries, rng, stop_on="in_query"):
    n, d = X.shape
    mu = np.zeros(d)
    Sigma = np.eye(d)
    used = set()
    log = []

    for step in range(max_queries):
        if stop_on == "argmax":
            diffs = X - mu
            Sigma_inv = np.linalg.inv(Sigma)
            d2 = np.einsum('nd,de,ne->n', diffs, Sigma_inv, diffs)
            if int(np.argmin(d2)) == target_idx:
                log.append({"step": step, "event": "stopped_argmax"})
                break

        i, j = sample_mirror(mu, Sigma, X, used, rng)
        used |= {i, j}

        # ground-truth Probit answer
        y_true = query_oracle(X[i], X[j], X[target_idx], sigma_eps, rng)

        # language round-trip
        picked = "A" if y_true == 0 else "B"
        utt, style = verbalize(picked, rng_trial)
        parsed = parse(utt)
        y_recovered, status = subset_to_y(parsed)

        entry = {
            "step": step,
            "query": (i, j),
            "classes": (labels[i], labels[j]),
            "y_true": y_true,
            "utterance": utt,
            "style": style,
            "parsed": parsed,
            "y_recovered": y_recovered,
            "status": status,
            "target_in_query": target_idx in (i, j),
        }

        if stop_on == "in_query" and target_idx in (i, j):
            log.append(entry)
            log.append({"step": step + 1, "event": "stopped_in_query"})
            break

        # apply update only on clean round-trip; otherwise skip
        if y_recovered is not None:
            mu, Sigma = adf_update(mu, Sigma, X[i], X[j], y_recovered, sigma_eps)
        entry["applied_update"] = (y_recovered is not None)
        log.append(entry)

    return log

In [25]:
# Cell 8 — run one search on target 437 and print the log
target_idx = 437
rng_trial = np.random.default_rng(seed=1000 + 40)

log = closed_loop_search(X, target_idx, SIGMA_EPS,
                          max_queries=50, rng=rng_trial, stop_on="in_query")

print(f"\ntarget: idx={target_idx} ({labels[target_idx]})\n" + "-"*80)
for e in log:
    if "event" in e:
        print(f"step {e['step']}: {e['event']}")
        continue
    star = "  *** TARGET ***" if e["target_in_query"] else ""
    match = "✓" if e["y_recovered"] == e["y_true"] else "✗" if e["y_recovered"] is not None else "-"
    print(f"step {e['step']:2d} {star}  [style={e['style']}]")
    print(f"  query: ({e['classes'][0]}, {e['classes'][1]})  y_true={e['y_true']}  "
          f"y_recovered={e['y_recovered']}  [{e['status']}] {match}")
    print(f"  utterance: {e['utterance']!r}")
    print(f"  parsed:    {e['parsed']!r}")


target: idx=437 (sea)
--------------------------------------------------------------------------------
step  0   [style=positional]
  query: (buildings, buildings)  y_true=1  y_recovered=0  [clean] ✗
  utterance: 'the right one'
  parsed:    ['A']
step  1   [style=casual]
  query: (street, buildings)  y_true=1  y_recovered=1  [clean] ✓
  utterance: 'Got B. Thanks!'
  parsed:    ['B']
step  2   [style=casual]
  query: (street, street)  y_true=1  y_recovered=1  [clean] ✓
  utterance: 'Thanks, got B.'
  parsed:    ['B']
step  3   [style=ordinal]
  query: (buildings, forest)  y_true=1  y_recovered=1  [clean] ✓
  utterance: 'the second one'
  parsed:    ['B']
step  4   [style=ordinal]
  query: (glacier, mountain)  y_true=0  y_recovered=0  [clean] ✓
  utterance: 'the first one'
  parsed:    ['A']
step  5   [style=ordinal]
  query: (glacier, sea)  y_true=0  y_recovered=0  [clean] ✓
  utterance: 'the first one'
  parsed:    ['A']
step  6   [style=ordinal]
  query: (mountain, mountain)  y_true

In [24]:
# Cell 9 — round-trip summary
def summarize(log):
    steps = [e for e in log if "event" not in e]
    n_steps = len(steps)
    n_clean = sum(1 for e in steps if e["status"] == "clean")
    n_correct = sum(1 for e in steps if e["y_recovered"] == e["y_true"])
    n_wrong = sum(1 for e in steps if e["y_recovered"] is not None and e["y_recovered"] != e["y_true"])
    n_skipped = sum(1 for e in steps if e["y_recovered"] is None)
    print(f"total steps:      {n_steps}")
    print(f"clean subset:     {n_clean} ({n_clean/n_steps:.0%})")
    print(f"round-trip match: {n_correct}/{n_steps} ({n_correct/n_steps:.0%})")
    print(f"parser flipped:   {n_wrong}")
    print(f"steps skipped:    {n_skipped}")
    from collections import Counter
    status_counts = Counter(e["status"] for e in steps)
    print(f"status breakdown: {dict(status_counts)}")

summarize(log)

total steps:      14
clean subset:     13 (93%)
round-trip match: 12/14 (86%)
parser flipped:   1
steps skipped:    1
status breakdown: {'clean': 13, 'both': 1}


In [26]:
# Cell 10 — pure Probit baseline over subsample of 100 targets
from tqdm.auto import tqdm

def pure_probit_search(X, target_idx, sigma_eps, max_queries, rng, stop_on="in_query"):
    n, d = X.shape
    mu = np.zeros(d)
    Sigma = np.eye(d)
    used = set()
    for step in range(max_queries):
        if stop_on == "argmax":
            diffs = X - mu
            Sigma_inv = np.linalg.inv(Sigma)
            d2 = np.einsum('nd,de,ne->n', diffs, Sigma_inv, diffs)
            if int(np.argmin(d2)) == target_idx:
                return step
        i, j = sample_mirror(mu, Sigma, X, used, rng)
        used |= {i, j}
        y = query_oracle(X[i], X[j], X[target_idx], sigma_eps, rng)
        if stop_on == "in_query" and target_idx in (i, j):
            return step + 1
        mu, Sigma = adf_update(mu, Sigma, X[i], X[j], y, sigma_eps)
    return max_queries

MAX_Q = 50
target_indices = list(range(0, n, 6))  # same subsample as closed loop
pure_results = []
for target_idx in tqdm(target_indices, desc="pure Probit"):
    rng_trial = np.random.default_rng(seed=1000 + target_idx)
    steps = pure_probit_search(X, target_idx, SIGMA_EPS, MAX_Q, rng_trial, "in_query")
    pure_results.append({"target_idx": target_idx, "class": labels[target_idx], "steps": steps})

print(f"pure Probit done: {len(pure_results)} targets")

pure Probit:   0%|          | 0/100 [00:00<?, ?it/s]

pure Probit done: 100 targets


In [ ]:
# Cell 11 — closed-loop over subsample of 100 targets
closed_results = []
target_indices = list(range(0, n, 6))  # every 6th → ~100 targets
for target_idx in tqdm(target_indices, desc="closed loop"):
    rng_trial = np.random.default_rng(seed=1000 + target_idx)
    log = closed_loop_search(X, target_idx, SIGMA_EPS, MAX_Q, rng_trial, "in_query")
    steps_entry = [e for e in log if "event" in e]
    steps = steps_entry[0]["step"] if steps_entry else MAX_Q
    step_entries = [e for e in log if "event" not in e]
    n_steps = len(step_entries)
    n_clean = sum(1 for e in step_entries if e["status"] == "clean")
    n_match = sum(1 for e in step_entries if e["y_recovered"] == e["y_true"])
    n_skip = sum(1 for e in step_entries if e["y_recovered"] is None)
    closed_results.append({
        "target_idx": target_idx,
        "class": labels[target_idx],
        "steps": steps,
        "n_step_entries": n_steps,
        "n_clean": n_clean,
        "n_match": n_match,
        "n_skip": n_skip,
    })

print(f"closed loop done: {len(closed_results)} targets")

closed loop:   0%|          | 0/100 [00:00<?, ?it/s]

In [29]:
# Cell 12 — analysis
import numpy as np
from collections import defaultdict

pure_steps = np.array([r["steps"] for r in pure_results])
closed_steps = np.array([r["steps"] for r in closed_results])

print("=== Overall step-count distribution ===")
print(f"                 pure Probit    closed loop")
print(f"  mean:          {pure_steps.mean():6.2f}         {closed_steps.mean():6.2f}")
print(f"  median:        {np.median(pure_steps):6.1f}         {np.median(closed_steps):6.1f}")
print(f"  p95:           {np.percentile(pure_steps, 95):6.1f}         {np.percentile(closed_steps, 95):6.1f}")
print(f"  max:           {pure_steps.max():6d}         {closed_steps.max():6d}")
print(f"  unconverged:   {(pure_steps >= MAX_Q).sum():6d}         {(closed_steps >= MAX_Q).sum():6d}")

overhead = (closed_steps.mean() / pure_steps.mean() - 1) * 100
print(f"\nlanguage overhead on mean: {overhead:+.1f}%")

# per-class breakdown
print("\n=== Per-class means ===")
print(f"  {'class':<10s}  pure    closed   overhead")
for cls in sorted(set(labels)):
    mask = np.array([r["class"] == cls for r in pure_results])
    p = pure_steps[mask].mean()
    c = closed_steps[mask].mean()
    ov = (c/p - 1) * 100
    print(f"  {cls:<10s}  {p:5.1f}   {c:5.1f}    {ov:+5.1f}%")

# round-trip stats
n_step_entries = sum(r["n_step_entries"] for r in closed_results)
n_clean = sum(r["n_clean"] for r in closed_results)
n_match = sum(r["n_match"] for r in closed_results)
n_skip = sum(r["n_skip"] for r in closed_results)
print(f"\n=== Round-trip stats (across all steps in closed-loop runs) ===")
print(f"  total step entries: {n_step_entries}")
print(f"  clean subset:       {n_clean} ({n_clean/n_step_entries:.1%})")
print(f"  round-trip match:   {n_match} ({n_match/n_step_entries:.1%})")
print(f"  skipped:            {n_skip} ({n_skip/n_step_entries:.1%})")
print(f"  parser flipped:     {n_step_entries - n_match - n_skip}")

=== Overall step-count distribution ===
                 pure Probit    closed loop
  mean:           15.10          19.16
  median:          15.0           17.0
  p95:             25.0           39.2
  max:               28             50
  unconverged:        0              2

language overhead on mean: +26.9%

=== Per-class means ===
  class       pure    closed   overhead
  buildings    14.1    21.4    +51.9%
  forest       17.5    20.3    +16.2%
  glacier      15.2    18.1    +18.9%
  mountain     13.6    15.4    +13.0%
  sea          15.4    18.4    +19.9%
  street       14.9    21.6    +44.4%

=== Round-trip stats (across all steps in closed-loop runs) ===
  total step entries: 1916
  clean subset:       1872 (97.7%)
  round-trip match:   1680 (87.7%)
  skipped:            44 (2.3%)
  parser flipped:     192


In [30]:
print(f"parser device: {next(parser.parameters()).device}")
print(f"verbalizer device: {next(verbalizer.parameters()).device}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

parser device: cuda:0
verbalizer device: cuda:0
VRAM used: 5.29 GB


In [31]:
# Cell — profile one step
import time

target_idx = 437
rng_trial = np.random.default_rng(seed=1000 + 40)

mu = np.zeros(d)
Sigma = np.eye(d)
used = set()

for step in range(3):
    print(f"\n--- step {step} ---")
    
    t0 = time.perf_counter()
    i, j = sample_mirror(mu, Sigma, X, used, rng_trial)
    used |= {i, j}
    t1 = time.perf_counter()
    print(f"  sample_mirror:  {(t1-t0)*1000:6.1f} ms")
    
    t0 = time.perf_counter()
    y_true = query_oracle(X[i], X[j], X[target_idx], SIGMA_EPS, rng_trial)
    t1 = time.perf_counter()
    print(f"  query_oracle:   {(t1-t0)*1000:6.1f} ms")
    
    t0 = time.perf_counter()
    picked = "A" if y_true == 0 else "B"
    utt, style = verbalize(picked, rng_trial)
    t1 = time.perf_counter()
    print(f"  verbalize:      {(t1-t0)*1000:6.1f} ms  ({len(utt.split())} words: {utt!r})")
    
    t0 = time.perf_counter()
    parsed = parse(utt)
    t1 = time.perf_counter()
    print(f"  parse:          {(t1-t0)*1000:6.1f} ms  (parsed: {parsed!r})")
    
    if parsed and isinstance(parsed, list) and len(parsed) == 1:
        y_rec = 0 if parsed[0] == "A" else 1
        t0 = time.perf_counter()
        mu, Sigma = adf_update(mu, Sigma, X[i], X[j], y_rec, SIGMA_EPS)
        t1 = time.perf_counter()
        print(f"  adf_update:     {(t1-t0)*1000:6.1f} ms")


--- step 0 ---
  sample_mirror:     0.7 ms
  query_oracle:      0.2 ms
  verbalize:      1668.0 ms  (3 words: 'the right one')
  parse:          1342.4 ms  (parsed: ['A'])
  adf_update:        1.1 ms

--- step 1 ---
  sample_mirror:     1.0 ms
  query_oracle:      0.2 ms
  verbalize:       674.0 ms  (3 words: 'Got B. Thanks!')
  parse:          1163.5 ms  (parsed: ['B'])
  adf_update:        1.0 ms

--- step 2 ---
  sample_mirror:     0.9 ms
  query_oracle:      0.3 ms
  verbalize:       557.1 ms  (3 words: 'Thanks, got B.')
  parse:          1155.7 ms  (parsed: ['B'])
  adf_update:        1.1 ms


In [ ]:
print(f"verbalizer model: {verbalizer.config._name_or_path if hasattr(verbalizer.config, '_name_or_path') else 'unknown'}")
print(f"verbalizer dtype: {verbalizer.dtype}")
print(f"parser model:     {parser.config._name_or_path if hasattr(parser.config, '_name_or_path') else 'unknown'}")
print(f"parser dtype:     {parser.dtype}")

In [15]:
import time

target_idx = 437
rng_trial = np.random.default_rng(seed=1000 + 40)

mu = np.zeros(d)
Sigma = np.eye(d)
used = set()

for step in range(5):
    print(f"\n--- step {step} ---")
    t0 = time.perf_counter()
    i, j = sample_mirror(mu, Sigma, X, used, rng_trial)
    used |= {i, j}
    t1 = time.perf_counter()
    print(f"  sample_mirror: {(t1-t0)*1000:6.1f} ms")
    t0 = time.perf_counter()
    y_true = query_oracle(X[i], X[j], X[target_idx], SIGMA_EPS, rng_trial)
    t1 = time.perf_counter()
    print(f"  query_oracle:  {(t1-t0)*1000:6.1f} ms")
    t0 = time.perf_counter()
    picked = "A" if y_true == 0 else "B"
    utt, style = verbalize(picked, rng_trial)
    t1 = time.perf_counter()
    print(f"  verbalize:     {(t1-t0)*1000:6.1f} ms  ({style}, {len(utt.split())} words)")
    t0 = time.perf_counter()
    parsed = parse(utt)
    t1 = time.perf_counter()
    print(f"  parse:         {(t1-t0)*1000:6.1f} ms  (parsed: {parsed!r})")
    if parsed and isinstance(parsed, list) and len(parsed) == 1:
        y_rec = 0 if parsed[0] == "A" else 1
        mu, Sigma = adf_update(mu, Sigma, X[i], X[j], y_rec, SIGMA_EPS)


--- step 0 ---
  sample_mirror:    0.6 ms
  query_oracle:     0.3 ms
  verbalize:      563.3 ms  (casual, 7 words)
  parse:          651.6 ms  (parsed: ['B'])

--- step 1 ---
  sample_mirror:    0.4 ms
  query_oracle:     0.1 ms
  verbalize:      246.5 ms  (question, 3 words)
  parse:          558.3 ms  (parsed: ['B'])

--- step 2 ---
  sample_mirror:    0.4 ms
  query_oracle:     0.3 ms
  verbalize:      640.9 ms  (hesitant, 12 words)
  parse:          587.0 ms  (parsed: ['A'])

--- step 3 ---
  sample_mirror:    0.2 ms
  query_oracle:     0.1 ms
  verbalize:      281.3 ms  (positional, 5 words)
  parse:          654.5 ms  (parsed: ['A'])

--- step 4 ---
  sample_mirror:    0.5 ms
  query_oracle:     0.1 ms
  verbalize:      268.6 ms  (ordinal, 5 words)
  parse:          636.6 ms  (parsed: ['A'])


In [33]:
# save current batch before rerunning
import pickle
with open("closed_loop_100_v2_no_logs.pkl", "wb") as f:
    pickle.dump({"pure": pure_results, "closed": closed_results}, f)
print("saved")

saved


In [34]:
# Cell 11 — closed-loop over subsample of 100 targets, WITH log saving
closed_results = []
closed_logs = []
target_indices = list(range(0, n, 6))
for target_idx in tqdm(target_indices, desc="closed loop"):
    rng_trial = np.random.default_rng(seed=1000 + target_idx)
    log = closed_loop_search(X, target_idx, SIGMA_EPS, MAX_Q, rng_trial, "in_query")
    steps_entry = [e for e in log if "event" in e]
    steps = steps_entry[0]["step"] if steps_entry else MAX_Q
    step_entries = [e for e in log if "event" not in e]
    n_steps = len(step_entries)
    n_clean = sum(1 for e in step_entries if e["status"] == "clean")
    n_match = sum(1 for e in step_entries if e["y_recovered"] == e["y_true"])
    n_skip = sum(1 for e in step_entries if e["y_recovered"] is None)
    closed_results.append({
        "target_idx": target_idx, "class": labels[target_idx], "steps": steps,
        "n_step_entries": n_steps, "n_clean": n_clean,
        "n_match": n_match, "n_skip": n_skip,
    })
    closed_logs.append({"target_idx": target_idx, "log": log})

print(f"done: {len(closed_results)} results, {len(closed_logs)} logs")

closed loop:   0%|          | 0/100 [00:00<?, ?it/s]

done: 100 results, 100 logs


In [35]:
# Cell 13 — per-style round-trip breakdown
from collections import defaultdict

style_stats = defaultdict(lambda: {"total": 0, "match": 0, "skip": 0, "flip": 0})
for tl in closed_logs:
    for e in tl["log"]:
        if "event" in e:
            continue
        s = e["style"]
        style_stats[s]["total"] += 1
        if e["y_recovered"] is None:
            style_stats[s]["skip"] += 1
        elif e["y_recovered"] == e["y_true"]:
            style_stats[s]["match"] += 1
        else:
            style_stats[s]["flip"] += 1

print(f"  {'style':<12s}  total  match%  skip%  flip%")
for style in sorted(style_stats.keys()):
    s = style_stats[style]
    t = s["total"]
    print(f"  {style:<12s}  {t:5d}  {s['match']/t:5.1%}  {s['skip']/t:5.1%}  {s['flip']/t:5.1%}")

  style         total  match%  skip%  flip%
  casual          368  87.8%  12.0%   0.3%
  direct          415  99.8%   0.0%   0.2%
  letter          415  100.0%   0.0%   0.0%
  ordinal         363  95.9%   0.0%   4.1%
  positional      355  50.7%   0.0%  49.3%


In [36]:
import pickle
with open("closed_loop_100_v3_with_logs.pkl", "wb") as f:
    pickle.dump({
        "pure": pure_results,
        "closed": closed_results,
        "logs": closed_logs,
        "style_stats": dict(style_stats),
    }, f)
print("saved")

saved
